In [5]:
import sys
from pathlib import Path
import torch

sys.path.append(str(Path("../src").resolve()))
from classifier_model import TrafficSignCNN

CLASSIFIER_PATH = Path("../models/gtsrb_cnn_v4_bn.pth")

classifier = TrafficSignCNN()
classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location="cpu"))
classifier.eval()
print("Classifier loaded")

Classifier loaded


In [6]:
from torchvision import transforms
from PIL import Image

preprocess = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

def classify(crop):
    x = preprocess(crop).unsqueeze(0)
    with torch.no_grad():
        logits = classifier(x)
        probs = torch.softmax(logits, dim=1)[0]
    conf, cls = probs.max(dim=0)
    return int(cls), float(conf)

In [7]:
SRC_DIR = Path("../data/FullIJCNN2013")

for class_folder in ["01", "14", "25", "38"]:
    img_path = sorted((SRC_DIR / class_folder).glob("*.ppm"))[0]
    with Image.open(img_path) as img:
        crop = img.convert("RGB")
    predicted, confidence = classify(crop)
    print(f"true class {int(class_folder)} -> predicted {predicted} (confidence {confidence:.2f})")

true class 1 -> predicted 1 (confidence 1.00)
true class 14 -> predicted 14 (confidence 1.00)
true class 25 -> predicted 25 (confidence 0.98)
true class 38 -> predicted 38 (confidence 1.00)


In [8]:
gt_per_image = {}

for line in (SRC_DIR / "gt.txt").read_text().splitlines():
    line = line.strip()
    if not line:
        continue
    filename, x1, y1, x2, y2, class_id = line.split(";")
    stem = Path(filename).stem
    if stem not in gt_per_image:
        gt_per_image[stem] = []
    gt_per_image[stem].append(((int(x1), int(y1), int(x2), int(y2)), int(class_id)))

In [9]:
VAL_IMAGES = Path("../data/yolo_dataset/images/val")

correct, total = 0, 0
mistakes = []

for img_path in sorted(VAL_IMAGES.glob("*.png")):
    signs = gt_per_image.get(img_path.stem, [])
    if not signs:
        continue
    with Image.open(img_path) as img:
        img = img.convert("RGB")
    for box, true_class in signs:
        crop = img.crop(box)
        predicted, confidence = classify(crop)
        total += 1
        if predicted == true_class:
            correct += 1
        else:
            mistakes.append((img_path.name, box, true_class, predicted, confidence))

print(f"Correct: {correct}/{total} = {correct / total:.3f}")

Correct: 178/184 = 0.967


In [10]:
CLASS_NAMES = [
    "Speed limit 20", "Speed limit 30", "Speed limit 50", "Speed limit 60",
    "Speed limit 70", "Speed limit 80", "End of speed limit 80", "Speed limit 100",
    "Speed limit 120", "No passing", "No passing for trucks",
    "Right-of-way at next intersection", "Priority road", "Yield", "Stop",
    "No vehicles", "Trucks prohibited", "No entry", "General caution",
    "Dangerous curve left", "Dangerous curve right", "Double curve", "Bumpy road",
    "Slippery road", "Road narrows on the right", "Road work", "Traffic signals",
    "Pedestrians", "Children crossing", "Bicycles crossing", "Beware of ice/snow",
    "Wild animals crossing", "End of all limits", "Turn right ahead",
    "Turn left ahead", "Ahead only", "Go straight or right", "Go straight or left",
    "Keep right", "Keep left", "Roundabout mandatory", "End of no passing",
    "End of no passing for trucks",
]
print(len(CLASS_NAMES))

43


In [11]:
for name, box, true_class, predicted, confidence in mistakes:
    w = box[2] - box[0]
    print(f"{name}: {CLASS_NAMES[true_class]} -> predicted {CLASS_NAMES[predicted]} "
          f"(confidence {confidence:.2f}, width {w} px)")

00011.png: Double curve -> predicted Right-of-way at next intersection (confidence 0.98, width 70 px)
00070.png: Slippery road -> predicted Dangerous curve right (confidence 0.53, width 80 px)
00358.png: Speed limit 120 -> predicted Speed limit 100 (confidence 0.80, width 58 px)
00369.png: Speed limit 120 -> predicted Speed limit 100 (confidence 0.78, width 21 px)
00478.png: Speed limit 80 -> predicted Speed limit 60 (confidence 0.94, width 42 px)
00607.png: Road narrows on the right -> predicted General caution (confidence 0.35, width 62 px)


***The full Pipeline***

In [15]:
from ultralytics import YOLO
from detection_utils import match_boxes

detector = YOLO("../models/yolo26n_1280_best.pt")
IMGSZ, CONF = 1280, 0.25

n_correct, n_misnamed, n_missed, n_phantom = 0, 0, 0, 0
pipeline_mistakes = []
classifications = []
phantom_confidences = []

for img_path in sorted(VAL_IMAGES.glob("*.png")):
    signs = gt_per_image.get(img_path.stem, [])
    true_boxes = [box for box, _ in signs]
    true_classes = [cls for _, cls in signs]

    r = detector.predict(img_path, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    pred_boxes = r.boxes.xyxy.tolist()
    pred_confs = r.boxes.conf.tolist()

    tp, fp, fn = match_boxes(true_boxes, pred_boxes, pred_confs)
    n_missed += len(fn)
    n_phantom += len(fp)

    if not tp and not fp:
        continue

    with Image.open(img_path) as img:
        img = img.convert("RGB")

    for i, j, _ in tp:
        crop = img.crop(tuple(round(v) for v in pred_boxes[i]))
        predicted, confidence = classify(crop)
        is_correct = predicted == true_classes[j]
        classifications.append((confidence, is_correct))
        if is_correct:
            n_correct += 1
        else:
            n_misnamed += 1
            pipeline_mistakes.append((img_path.name, true_classes[j], predicted, confidence))

    for i in fp:
        crop = img.crop(tuple(round(v) for v in pred_boxes[i]))
        _, confidence = classify(crop)
        phantom_confidences.append(confidence)

total = n_correct + n_misnamed + n_missed
print(f"True signs: {total}")
print(f"Correct:  {n_correct} ({n_correct / total:.3f})")
print(f"Misnamed: {n_misnamed}")
print(f"Missed:   {n_missed}")
print(f"Phantom boxes: {n_phantom}")

True signs: 184
Correct:  175 (0.951)
Misnamed: 7
Missed:   2
Phantom boxes: 7


In [13]:
perfect_wrong = {(name, true_class) for name, box, true_class, predicted, conf in mistakes}
pipeline_wrong = {(name, true_class) for name, true_class, predicted, conf in pipeline_mistakes}

print("Wrong only with YOLO crops:", pipeline_wrong - perfect_wrong)
print("Wrong only with perfect boxes:", perfect_wrong - pipeline_wrong)

Wrong only with YOLO crops: {('00662.png', 22), ('00311.png', 8), ('00624.png', 2)}
Wrong only with perfect boxes: {('00011.png', 21), ('00607.png', 24)}


In [14]:
for name, true_class, predicted, confidence in pipeline_mistakes:
    print(f"{name}: {CLASS_NAMES[true_class]} -> {CLASS_NAMES[predicted]} ({confidence:.2f})")

00070.png: Slippery road -> Dangerous curve right (0.51)
00311.png: Speed limit 120 -> Speed limit 100 (0.80)
00358.png: Speed limit 120 -> Speed limit 100 (0.74)
00369.png: Speed limit 120 -> Speed limit 100 (0.95)
00478.png: Speed limit 80 -> Speed limit 60 (0.94)
00624.png: Speed limit 50 -> Speed limit 30 (0.51)
00662.png: Bumpy road -> Bicycles crossing (0.61)


In [16]:
print([round(c, 2) for c in phantom_confidences])


[0.89, 0.61, 0.43, 0.82, 0.5, 0.65, 0.79]


In [17]:
print(f"{'T':>5} | correct kept | correct lost | wrong kept | wrong caught | phantoms named | phantoms caught")

for T in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    correct_kept = sum(1 for c, ok in classifications if ok and c >= T)
    correct_lost = sum(1 for c, ok in classifications if ok and c < T)
    wrong_kept = sum(1 for c, ok in classifications if not ok and c >= T)
    wrong_caught = sum(1 for c, ok in classifications if not ok and c < T)
    phantoms_named = sum(1 for c in phantom_confidences if c >= T)
    phantoms_caught = sum(1 for c in phantom_confidences if c < T)

    print(f"{T:>5} | {correct_kept:>12} | {correct_lost:>12} | {wrong_kept:>10} | "
          f"{wrong_caught:>12} | {phantoms_named:>14} | {phantoms_caught:>15}")

    T | correct kept | correct lost | wrong kept | wrong caught | phantoms named | phantoms caught
  0.0 |          175 |            0 |          7 |            0 |              7 |               0
  0.5 |          175 |            0 |          7 |            0 |              6 |               1
  0.6 |          175 |            0 |          5 |            2 |              5 |               2
  0.7 |          173 |            2 |          4 |            3 |              3 |               4
  0.8 |          172 |            3 |          2 |            5 |              2 |               5
  0.9 |          164 |           11 |          2 |            5 |              0 |               7
 0.95 |          157 |           18 |          1 |            6 |              0 |               7


## Final evaluation: pipeline with the ‘unknown’ rule (T = 0.8)

In [18]:
UNKNOWN_T = 0.8

def run_pipeline(image_dir, threshold):
    counts = {"correct": 0, "wrong": 0, "unknown": 0, "missed": 0,
              "phantom_named": 0, "phantom_unknown": 0}

    for img_path in sorted(image_dir.glob("*.png")):
        signs = gt_per_image.get(img_path.stem, [])
        true_boxes = [box for box, _ in signs]
        true_classes = [cls for _, cls in signs]

        r = detector.predict(img_path, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
        pred_boxes = r.boxes.xyxy.tolist()
        pred_confs = r.boxes.conf.tolist()

        tp, fp, fn = match_boxes(true_boxes, pred_boxes, pred_confs)
        counts["missed"] += len(fn)

        if not tp and not fp:
            continue

        with Image.open(img_path) as img:
            img = img.convert("RGB")

        for i, j, _ in tp:
            predicted, confidence = classify(img.crop(tuple(round(v) for v in pred_boxes[i])))
            if confidence < threshold:
                counts["unknown"] += 1
            elif predicted == true_classes[j]:
                counts["correct"] += 1
            else:
                counts["wrong"] += 1

        for i in fp:
            _, confidence = classify(img.crop(tuple(round(v) for v in pred_boxes[i])))
            if confidence < threshold:
                counts["phantom_unknown"] += 1
            else:
                counts["phantom_named"] += 1

    return counts

In [19]:
val_counts = run_pipeline(VAL_IMAGES, UNKNOWN_T)
print(val_counts)

{'correct': 172, 'wrong': 2, 'unknown': 8, 'missed': 2, 'phantom_named': 2, 'phantom_unknown': 5}


In [20]:
TEST_IMAGES = Path("../data/yolo_dataset/images/test")
test_counts = run_pipeline(TEST_IMAGES, UNKNOWN_T)

n_signs = test_counts["correct"] + test_counts["wrong"] + test_counts["unknown"] + test_counts["missed"]
print(test_counts)
print(f"True signs: {n_signs}")
print(f"Correctly named: {test_counts['correct'] / n_signs:.3f}")
print(f"Wrong sign shown: {test_counts['wrong'] / n_signs:.3f}")
print(f"Unknown: {test_counts['unknown'] / n_signs:.3f}")
print(f"Missed: {test_counts['missed'] / n_signs:.3f}")

{'correct': 182, 'wrong': 0, 'unknown': 14, 'missed': 4, 'phantom_named': 8, 'phantom_unknown': 9}
True signs: 200
Correctly named: 0.910
Wrong sign shown: 0.000
Unknown: 0.070
Missed: 0.020
